# Okanka Mini — 03: SFT (chat format + Turkish instructions + identity)

Run after notebook 02 has reached its final step. Settings: GPU T4 x2, Internet On. ~0.5-1.5 h.

nanochat's current version has no separate mid-training stage (it was merged into SFT); this notebook runs
`scripts.chat_sft` with the Turkish mixture (`--mixture tr`): Aya-Turkish + OASST2-Turkish + Wikipedia template QA
(x2 epochs) + identity conversations (x4). The result is pushed to `<HF_USERNAME>/okanka-mini-checkpoints/chatsft_checkpoints/d<DEPTH>`.

In [ ]:
# ======================= OKANKA MINI CONFIG (edit if needed) =======================
REPO_URL   = "https://github.com/alisimseksembol/okanka-mini.git"   # GitHub repo with this code (public, or add Kaggle secret GITHUB_TOKEN)
BRANCH     = "main"
DEPTH      = 16          # model depth (d16 ~ 235M "scaling" params). Use 12 for a faster/cheaper run.
SEQ_LEN    = 1024        # context length used for pretraining on T4
DEVICE_BS  = 4 if DEPTH >= 14 else 8   # per-GPU micro batch (power of 2; halve it if CUDA OOM)
DATA_RATIO = 12          # tokens : scaling-params ratio (nanochat compute-optimal ~10-12; Chinchilla 20)
# Hugging Face repos (created automatically, private):
#   <HF_USERNAME>/okanka-mini-data         (dataset: pretraining shards + SFT jsonl)
#   <HF_USERNAME>/okanka-mini-checkpoints  (model: resumable training checkpoints)
#   <HF_USERNAME>/okanka-mini              (model: final public export, notebook 04)
# ===================================================================================

In [ ]:
import os, sys, subprocess, shutil
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name, default=None):
    try:
        return secrets.get_secret(name)
    except Exception:
        return default

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Add a Kaggle secret named HF_TOKEN (HF write token): Add-ons -> Secrets"
HF_USERNAME = secret("HF_USERNAME")
if not HF_USERNAME:
    from huggingface_hub import whoami
    HF_USERNAME = whoami(token=os.environ["HF_TOKEN"])["name"]
DATA_REPO = f"{HF_USERNAME}/okanka-mini-data"
CKPT_REPO = f"{HF_USERNAME}/okanka-mini-checkpoints"
FINAL_REPO = f"{HF_USERNAME}/okanka-mini"
print("HF user:", HF_USERNAME)

# --- code ---
gh = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
CODE = "/kaggle/working/okanka-mini"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull"], check=False)
os.chdir(CODE)
sys.path.insert(0, CODE)

# --- python deps (torch comes with the Kaggle image) ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken>=0.11.0", "rustbpe>=0.1.0", "pyarrow>=21.0.0",
                "filelock", "psutil", "huggingface_hub>=0.34.0", "wandb"], check=True)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
major, minor = (int(x) for x in torch.__version__.split(".")[:2])
if (major, minor) < (2, 6):
    # nanochat needs a recent torch (SDPA enable_gqa, F.rms_norm, compile fixes). cu126 wheels still support T4 (sm75) and P100 (sm60).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
    print("Upgraded torch -> restart the kernel (Run > Restart) and run all cells again."); raise SystemExit

# --- nanochat runtime env ---
os.environ["NANOCHAT_BASE_DIR"] = "/tmp/nanochat"           # big ephemeral disk; persistent state lives on the HF hub
os.environ["NANOCHAT_DISABLE_FA3"] = "1"                    # FA3 is Hopper/Ampere-only; T4 uses PyTorch SDPA
os.environ["OMP_NUM_THREADS"] = "1"
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap >= (8, 0):
        os.environ["NANOCHAT_DTYPE"] = "bfloat16"
    elif cap >= (7, 0):
        os.environ["NANOCHAT_DTYPE"] = "float16"            # T4/V100: fp16 tensor cores + GradScaler (nanochat built-in)
    else:
        os.environ["NANOCHAT_DTYPE"] = "float32"            # P100: no tensor cores, triton unsupported -> no compile
        os.environ["TORCHDYNAMO_DISABLE"] = "1"
NGPU = max(1, torch.cuda.device_count())
print("NANOCHAT_DTYPE =", os.environ.get("NANOCHAT_DTYPE"), "| GPUs:", NGPU)
os.makedirs(os.environ["NANOCHAT_BASE_DIR"], exist_ok=True)

def run(cmd):
    """Run a shell command, streaming its output into the notebook; raise on failure."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed ({rc}): {cmd}")

In [ ]:
from okanka import hf_sync
from huggingface_hub import hf_hub_download
step = hf_sync.pull(CKPT_REPO, f"base_checkpoints/d{DEPTH}", os.environ["NANOCHAT_BASE_DIR"])
assert step is not None, "no pretraining checkpoint found - run notebook 02 first"
print("base step:", step)
os.makedirs("data/sft", exist_ok=True)
for f in ["sft/sft_tr_train.jsonl", "sft/sft_tr_val.jsonl", "sft/identity_tr.jsonl"]:
    p = hf_hub_download(DATA_REPO, f, repo_type="dataset", token=os.environ["HF_TOKEN"])
    shutil.copy(p, "data/" + (f if "identity" not in f else "identity_tr.jsonl"))
!wc -l data/sft/*.jsonl data/identity_tr.jsonl

In [ ]:
compile_flag = "--no-compile" if os.environ.get("TORCHDYNAMO_DISABLE") == "1" else ""
run(f"torchrun --standalone --nproc_per_node={NGPU} -m scripts.chat_sft -- --mixture=tr "
    f"--device-batch-size={DEVICE_BS} --total-batch-size={DEVICE_BS*SEQ_LEN*NGPU*4} "
    f"--eval-every=100 --eval-tokens={10*SEQ_LEN*DEVICE_BS*NGPU} --sft-epochs=2 --identity-epochs=4 {compile_flag}")

In [ ]:
from nanochat.checkpoint_manager import find_last_step
sft_dir = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "chatsft_checkpoints", f"d{DEPTH}")
s = find_last_step(sft_dir)
hf_sync.push(CKPT_REPO, f"chatsft_checkpoints/d{DEPTH}", s, os.environ["NANOCHAT_BASE_DIR"])
print("SFT step", s, "pushed")

In [ ]:
# quick smoke test
run('python -m scripts.chat_cli -i sft -p "Merhaba! Sen kimsin?" -t 0.6')